# EDA — NVIDIA Nemotron Model Reasoning Challenge

Public data: **9,500 train** rows × `(id, prompt, answer)` and **3 test** rows (the real test set is hidden on the leaderboard and is "several hundred" prompts).

Every prompt is a "In Alice's Wonderland, ..." few-shot rule-induction puzzle. A quick scan reveals exactly **six puzzle families**:

| Family | Task |
|---|---|
| `bit_manip` | 8-bit binary transformation (induce XOR / shift / rotate / NOT rule) |
| `gravity` | Infer secret `g` from `(t, d)` pairs, then apply `d = 0.5·g·t²` |
| `unit_conv` | Infer secret linear unit-conversion factor |
| `encryption` | Monoalphabetic substitution cipher on natural-language text |
| `numeral` | Roman (or Roman-like) numeral system |
| `equations` | Character-level substitution over a small symbolic alphabet |

This notebook confirms the taxonomy, checks prompt/answer length distributions, and previews each family.

In [ ]:
import sys
from pathlib import Path

# Make project root importable when running the notebook from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.taxonomy import FAMILIES, classify_many, summarize

pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

In [ ]:
train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
print("train:", train.shape, list(train.columns))
print("test: ", test.shape, list(test.columns))
train.head(3)

## Puzzle family distribution

Assign every train prompt to one of the six canonical families using `src.data.taxonomy.classify`. We expect near-uniform counts (~1,575 per family) and 0% unknowns.

In [ ]:
train["family"] = classify_many(train["prompt"].tolist())
report = summarize(train["prompt"].tolist())
print("unknown rate:", f"{report.unknown_rate:.2%}")
counts = pd.Series(report.counts).reindex(FAMILIES).dropna()
print(counts)

fig, ax = plt.subplots(figsize=(8, 3.5))
counts.drop(labels=[x for x in ["unknown"] if x in counts.index]).plot.bar(ax=ax, color="#76b900")
ax.set_title("Puzzle family counts (train)")
ax.set_ylabel("rows")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Prompt length (characters)

Prompts are short. Under the 30B base-model tokenizer we'll stay well under the 8,192-token `max_model_len`, leaving ~7.5k tokens of headroom for CoT reasoning at inference time.

In [ ]:
train["prompt_chars"] = train["prompt"].str.len()
print(train.groupby("family")["prompt_chars"].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for fam, sub in train.groupby("family"):
    ax.hist(sub["prompt_chars"], bins=40, alpha=0.45, label=fam)
ax.set_xlabel("prompt length (chars)")
ax.set_ylabel("count")
ax.set_title("Prompt-length distribution by family")
ax.legend()
plt.tight_layout()
plt.show()

## Answer type analysis

Answers are short and strongly typed per family (binary strings, floats with 2dp, Roman numerals, symbol strings). This informs: (a) how to construct synthetic generators, and (b) how to format the `\boxed{...}` output during SFT.

In [ ]:
train["answer_chars"] = train["answer"].astype(str).str.len()
stats = (
    train.groupby("family")
    .agg(
        n=("answer", "size"),
        unique_answers=("answer", "nunique"),
        ans_chars_mean=("answer_chars", "mean"),
        ans_chars_max=("answer_chars", "max"),
    )
    .round(2)
)
print(stats)

for fam in [f for f in FAMILIES if f != "unknown"]:
    sub = train[train["family"] == fam].head(2)
    print(f"\n--- {fam} ---")
    for _, row in sub.iterrows():
        print(row["prompt"][:280].replace("\n", " \\n "))
        print("  ANSWER:", repr(row["answer"]))

## Test set preview

The public `test.csv` has only 3 placeholder rows. The real leaderboard test set is hidden.

In [ ]:
for _, row in test.iterrows():
    print("id:", row["id"])
    print(row["prompt"])
    print("---")

## Takeaways

- **6 clean, parametric families** — every row can be deterministically tagged; no unknowns.
- **Short prompts (~300 chars, max ~510)** — massive CoT headroom under the 8,192-token inference budget.
- **Typed answers** per family → we will build 6 verifiable synthetic generators (rule-sampler → N examples → hidden answer) to scale training data beyond the 9.5k public rows.
- **Uniform family balance** → stratify SFT/GRPO sampling and local eval splits by `family` column.
- **Hidden test is larger but same format** → no domain shift expected; overfitting to train-specific secret rules is a real risk → synthetic data + GRPO should dominate pure SFT.